In [20]:

import re
import json
import csv
from pathlib import Path
from datetime import datetime

print("Setup complete")



Setup complete


In [21]:

PLANNER_PROMPT = """
You are the Planner in a multi-step reasoning agent.

Read the question and return a concise JSON plan with:
- problem_type
- required facts and quantities
- steps
- expected answer format

Do not provide a long chain of thought. Return valid JSON only.

Examples:
Question: A train leaves at 14:30 and arrives at 18:05.
Plan: {"problem_type":"time_difference",
       "steps":["Parse departure and arrival times",
                "Calculate elapsed minutes",
                "Format hours and minutes"]}

Question: Alice has 3 red apples and twice as many green apples.
Plan: {"problem_type":"arithmetic",
       "steps":["Count red apples","Calculate green apples",
                "Add both quantities"]}

Question: A meeting needs 60 minutes and a slot lasts 30 minutes.
Plan: {"problem_type":"constraint_check",
       "steps":["Calculate available minutes",
                "Compare available and required time"]}
"""

EXECUTOR_PROMPT = """
You are the Executor. Follow the supplied plan and solve the question.

Return valid JSON with:
- answer: short final answer
- explanation: brief user-facing explanation
- intermediate_result: key calculation or result

Do not invent missing information. Do not expose private chain-of-thought.
Check units and use the question's stated quantities.

Examples:
Question: 5 pens cost 10 rupees each.
Output: {"answer":"50 rupees",
         "explanation":"5 x 10 = 50 rupees.",
         "intermediate_result":"5 * 10 = 50"}

Question: A journey starts at 14:30 and ends at 18:05.
Output: {"answer":"3 hours 35 minutes",
         "explanation":"The elapsed time is 215 minutes.",
         "intermediate_result":"215 minutes"}

Question: A meeting needs 60 minutes but the slot is 30 minutes.
Output: {"answer":"No",
         "explanation":"The slot is 30 minutes, less than 60 minutes.",
         "intermediate_result":"30 < 60"}
"""

VERIFIER_PROMPT = """
You are the Verifier. Independently check the proposed answer against
the question and calculations. Check arithmetic, units, time boundaries,
constraints, and consistency between the answer and explanation.

Return valid JSON:
{"passed":true_or_false,
 "check_name":"string",
 "details":"short explanation",
 "correction":"correct answer if needed"}

Examples:
Question: 3 red apples and twice as many green apples.
Proposed answer: 9 apples.
Output: {"passed":true,"check_name":"arithmetic",
         "details":"3 + 6 = 9.","correction":""}

Question: 12 chocolates, eat 4, buy 3.
Proposed answer: 8 chocolates.
Output: {"passed":false,"check_name":"arithmetic",
         "details":"12 - 4 + 3 = 11.","correction":"11 chocolates"}

Question: Need 60 minutes, slot has 30 minutes.
Proposed answer: Yes.
Output: {"passed":false,"check_name":"constraints",
         "details":"30 minutes is insufficient.","correction":"No"}
"""

print("Planner, Executor, and Verifier prompts defined.")


Planner, Executor, and Verifier prompts defined.


In [22]:

def parse_time(value):
    return datetime.strptime(value, "%H:%M")


def planner(question):
    q = question.lower()

    # Meeting duration and available slots
    if "meeting" in q and ("slot" in q or "free" in q):
        duration_match = re.search(
            r"(\d+)\s*minutes?", q
        )
        slots = re.findall(
            r"(\d{1,2}:\d{2})\s*[-–—]\s*(\d{1,2}:\d{2})",
            q
        )

        if duration_match and slots:
            return {
                "problem_type": "meeting_slots",
                "required_minutes": int(duration_match.group(1)),
                "slots": slots,
                "steps": [
                    "Calculate the duration of each slot.",
                    "Compare each duration with the required duration.",
                    "Return slots that meet or exceed the requirement."
                ]
            }

    # Journey duration
    if "train" in q or "journey" in q:
        times = re.findall(r"\b\d{1,2}:\d{2}\b", q)
        if len(times) >= 2:
            return {
                "problem_type": "journey",
                "departure": times[0],
                "arrival": times[1],
                "steps": [
                    "Parse departure and arrival times.",
                    "Calculate elapsed time.",
                    "Format the duration."
                ]
            }

    # Apples: red apples and twice as many green apples
    if "apple" in q:
        red = re.search(r"(\d+)\s+red apples?", q)
        if red and ("twice" in q or "double" in q):
            return {
                "problem_type": "apples",
                "red": int(red.group(1)),
                "green_multiplier": 2,
                "steps": [
                    "Extract the number of red apples.",
                    "Calculate the number of green apples.",
                    "Add the two quantities."
                ]
            }

    # Chocolates: initial amount, eaten, and bought
    if "chocolate" in q:
        initial = re.search(r"(\d+)\s+chocolates?", q)
        eaten = re.search(r"eats?\s+(\d+)", q)
        bought = re.search(r"buys?\s+(\d+)", q)

        if initial and eaten and bought:
            return {
                "problem_type": "chocolates",
                "initial": int(initial.group(1)),
                "eaten": int(eaten.group(1)),
                "bought": int(bought.group(1)),
                "steps": [
                    "Subtract the eaten chocolates.",
                    "Add the bought chocolates."
                ]
            }

    # Price calculation
    if "pen" in q:
        quantity = re.search(r"(\d+)\s+pens?", q)
        price = re.search(r"(\d+)\s+rupees?\s+each", q)

        if quantity and price:
            return {
                "problem_type": "price",
                "quantity": int(quantity.group(1)),
                "price_each": int(price.group(1)),
                "steps": ["Multiply quantity by unit price."]
            }

    return {
        "problem_type": "unsupported",
        "steps": [
            "The offline parser does not recognize this question format."
        ]
    }


print("Planner ready.")


Planner ready.


In [23]:

def executor(question, plan):
    kind = plan["problem_type"]

    if kind == "apples":
        red = plan["red"]
        green = red * plan["green_multiplier"]
        total = red + green

        return {
            "answer": f"{total} apples",
            "explanation": (
                f"There are {red} red apples and {green} green apples, "
                f"so the total is {total}."
            ),
            "value": total
        }

    if kind == "chocolates":
        remaining = plan["initial"] - plan["eaten"]
        total = remaining + plan["bought"]

        return {
            "answer": f"{total} chocolates",
            "explanation": (
                f"After eating {plan['eaten']}, {remaining} remain. "
                f"After buying {plan['bought']} more, the total is {total}."
            ),
            "value": total
        }

    if kind == "price":
        total = plan["quantity"] * plan["price_each"]

        return {
            "answer": f"{total} rupees",
            "explanation": (
                f"{plan['quantity']} pens at "
                f"{plan['price_each']} rupees each cost {total} rupees."
            ),
            "value": total
        }

    if kind == "journey":
        start = parse_time(plan["departure"])
        end = parse_time(plan["arrival"])
        minutes = int((end - start).total_seconds() / 60)

        # Handle a journey that arrives after midnight.
        if minutes < 0:
            minutes += 24 * 60

        hours, remaining = divmod(minutes, 60)
        hour_word = "hour" if hours == 1 else "hours"
        answer = f"{hours} {hour_word} {remaining} minutes"

        return {
            "answer": answer,
            "explanation": f"The journey lasts {minutes} minutes.",
            "value": minutes
        }

    if kind == "meeting_slots":
        required = plan["required_minutes"]
        suitable = []

        for start_text, end_text in plan["slots"]:
            start = parse_time(start_text)
            end = parse_time(end_text)
            duration = int((end - start).total_seconds() / 60)

            if duration >= required:
                suitable.append(f"{start_text}–{end_text}")

        answer = ", ".join(suitable) if suitable else "No suitable slots"

        return {
            "answer": answer,
            "explanation": (
                f"Selected slots have at least {required} minutes available."
                if suitable else
                f"No slot is at least {required} minutes long."
            ),
            "value": suitable
        }

    return {
        "answer": "Unsupported question",
        "explanation": "The offline agent cannot solve this question format.",
        "value": None
    }


print("Executor ready.")


Executor ready.


In [24]:

def verifier(question, plan, solution):
    kind = plan["problem_type"]
    actual = solution.get("value")

    if kind == "apples":
        expected = plan["red"] + (
            plan["red"] * plan["green_multiplier"]
        )
        passed = actual == expected
        details = f"Expected total: {expected} apples."

    elif kind == "chocolates":
        expected = (
            plan["initial"] - plan["eaten"] + plan["bought"]
        )
        passed = actual == expected
        details = f"Expected total: {expected} chocolates."

    elif kind == "price":
        expected = plan["quantity"] * plan["price_each"]
        passed = actual == expected
        details = f"Expected cost: {expected} rupees."

    elif kind == "journey":
        start = parse_time(plan["departure"])
        end = parse_time(plan["arrival"])
        expected = int((end - start).total_seconds() / 60)

        if expected < 0:
            expected += 24 * 60

        passed = actual == expected
        details = f"Expected elapsed time: {expected} minutes."

    elif kind == "meeting_slots":
        expected = []

        for start_text, end_text in plan["slots"]:
            start = parse_time(start_text)
            end = parse_time(end_text)
            duration = int((end - start).total_seconds() / 60)

            if duration >= plan["required_minutes"]:
                expected.append(f"{start_text}–{end_text}")

        passed = actual == expected
        details = (
            f"Expected suitable slots: "
            f"{', '.join(expected) if expected else 'none'}."
        )

    else:
        passed = False
        details = "Unsupported problem type; cannot verify."

    return {
        "passed": bool(passed),
        "check_name": "independent_calculation_and_constraints",
        "details": details
    }


print("Verifier ready.")


Verifier ready.


In [25]:

MAX_RETRIES = 2

def solve(question, executor_fn=executor, max_retries=MAX_RETRIES):
    checks = []
    retries = 0
    plan = {}

    try:
        plan = planner(question)

        while True:
            solution = executor_fn(question, plan)
            check = verifier(question, plan, solution)

            checks.append({
                "check_name": check["check_name"],
                "passed": check["passed"],
                "details": check["details"]
            })

            if check["passed"]:
                return {
                    "answer": solution["answer"],
                    "status": "success",
                    "reasoning_visible_to_user": solution["explanation"],
                    "metadata": {
                        "plan": json.dumps(plan),
                        "checks": checks,
                        "retries": retries
                    }
                }

            if retries >= max_retries:
                return {
                    "answer": "Unable to verify a correct answer.",
                    "status": "failed",
                    "reasoning_visible_to_user":
                        "The solution failed verification after the permitted attempts.",
                    "metadata": {
                        "plan": json.dumps(plan),
                        "checks": checks,
                        "retries": retries
                    }
                }

            retries += 1

    except Exception as e:
        return {
            "answer": "Unable to solve the question.",
            "status": "failed",
            "reasoning_visible_to_user":
                "An error occurred during processing.",
            "metadata": {
                "plan": json.dumps(plan) if plan else "",
                "checks": checks,
                "retries": retries,
                "error": str(e)
            }
        }


print("Complete agent ready.")


Complete agent ready.


In [26]:

question = (
    "Alice has 3 red apples and twice as many green apples as red. "
    "How many apples does she have in total?"
)

result = solve(question)
print(json.dumps(result, indent=2))


{
  "answer": "9 apples",
  "status": "success",
  "reasoning_visible_to_user": "There are 3 red apples and 6 green apples, so the total is 9.",
  "metadata": {
    "plan": "{\"problem_type\": \"apples\", \"red\": 3, \"green_multiplier\": 2, \"steps\": [\"Extract the number of red apples.\", \"Calculate the number of green apples.\", \"Add the two quantities.\"]}",
    "checks": [
      {
        "check_name": "independent_calculation_and_constraints",
        "passed": true,
        "details": "Expected total: 9 apples."
      }
    ],
    "retries": 0
  }
}


In [27]:

easy_tests = [
    {
        "question": "Alice has 3 red apples and twice as many green apples as red. How many apples does she have in total?",
        "expected": "9 apples"
    },
    {
        "question": "Alice has 5 red apples and twice as many green apples as red. How many apples does she have in total?",
        "expected": "15 apples"
    },
    {
        "question": "There are 12 chocolates. Ravi eats 4 and buys 3 more. How many does he have?",
        "expected": "11 chocolates"
    },
    {
        "question": "There are 20 chocolates. Ravi eats 7 and buys 2 more. How many does he have?",
        "expected": "15 chocolates"
    },
    {
        "question": "A shop sells 5 pens for 10 rupees each. What is the total cost?",
        "expected": "50 rupees"
    },
    {
        "question": "A shop sells 8 pens for 5 rupees each. What is the total cost?",
        "expected": "40 rupees"
    },
    {
        "question": "A train leaves at 14:30 and arrives at 18:05. How long is the journey?",
        "expected": "3 hours 35 minutes"
    },
    {
        "question": "A train leaves at 09:15 and arrives at 10:45. How long is the journey?",
        "expected": "1 hour 30 minutes"
    }
]

tricky_tests = [
    {
        "question": "A meeting needs 60 minutes. There are free slots: 09:00–09:30, 09:45–10:30, 11:00–12:00. Which slots can fit the meeting?",
        "expected": "11:00–12:00"
    },
    {
        "question": "A train leaves at 23:30 and arrives at 00:15. How long is the journey?",
        "expected": "0 hours 45 minutes"
    },
    {
        "question": "A meeting needs 30 minutes. There are free slots: 09:00–09:30, 10:00–10:29. Which slots can fit the meeting?",
        "expected": "09:00–09:30"
    },
    {
        "question": "John has some apples and gives away some of them. How many apples remain?",
        "expected_status": "failed"
    }
]

print("Easy tests:", len(easy_tests))
print("Tricky tests:", len(tricky_tests))
print("Total tests:", len(easy_tests) + len(tricky_tests))


Easy tests: 8
Tricky tests: 4
Total tests: 12


In [28]:

all_tests = [
    ("easy", test) for test in easy_tests
] + [
    ("tricky", test) for test in tricky_tests
]

logs = []
passed_count = 0

for category, test in all_tests:
    result = solve(test["question"])

    expected_status = test.get("expected_status", "success")
    if expected_status == "failed":
        test_passed = result["status"] == "failed"
    else:
        test_passed = (
            result["status"] == "success"
            and result["answer"] == test["expected"]
        )

    if test_passed:
        passed_count += 1

    checks = result["metadata"]["checks"]
    verifier_passed = bool(checks) and checks[-1]["passed"]

    log_entry = {
        "category": category,
        "question": test["question"],
        "expected": test.get("expected", expected_status),
        "final_json": result,
        "verifier_passed": verifier_passed,
        "retries": result["metadata"]["retries"],
        "test_passed": test_passed
    }

    logs.append(log_entry)

    print("=" * 70)
    print("Category:", category)
    print("Question:", test["question"])
    print(json.dumps(result, indent=2))
    print("Verifier passed:", verifier_passed)
    print("Retries:", result["metadata"]["retries"])
    print("Test passed:", test_passed)

print("\nFINAL SCORE:", passed_count, "/", len(all_tests))

with open("example_run_logs.json", "w", encoding="utf-8") as file:
    json.dump(logs, file, indent=2, ensure_ascii=False)

print("Saved logs to example_run_logs.json")


Category: easy
Question: Alice has 3 red apples and twice as many green apples as red. How many apples does she have in total?
{
  "answer": "9 apples",
  "status": "success",
  "reasoning_visible_to_user": "There are 3 red apples and 6 green apples, so the total is 9.",
  "metadata": {
    "plan": "{\"problem_type\": \"apples\", \"red\": 3, \"green_multiplier\": 2, \"steps\": [\"Extract the number of red apples.\", \"Calculate the number of green apples.\", \"Add the two quantities.\"]}",
    "checks": [
      {
        "check_name": "independent_calculation_and_constraints",
        "passed": true,
        "details": "Expected total: 9 apples."
      }
    ],
    "retries": 0
  }
}
Verifier passed: True
Retries: 0
Test passed: True
Category: easy
Question: Alice has 5 red apples and twice as many green apples as red. How many apples does she have in total?
{
  "answer": "15 apples",
  "status": "success",
  "reasoning_visible_to_user": "There are 5 red apples and 10 green apples, so 

In [29]:

retry_state = {"calls": 0}

def faulty_once_executor(question, plan):
    retry_state["calls"] += 1

    if retry_state["calls"] == 1:
        return {
            "answer": "8 apples",
            "explanation": "Intentionally incorrect answer for retry testing.",
            "value": 8
        }

    return executor(question, plan)


retry_result = solve(
    "Alice has 3 red apples and twice as many green apples as red. How many apples does she have in total?",
    executor_fn=faulty_once_executor,
    max_retries=2
)

print(json.dumps(retry_result, indent=2))

assert retry_result["status"] == "success"
assert retry_result["answer"] == "9 apples"
assert retry_result["metadata"]["retries"] == 1
assert len(retry_result["metadata"]["checks"]) == 2

print("Retry test passed!")


{
  "answer": "9 apples",
  "status": "success",
  "reasoning_visible_to_user": "There are 3 red apples and 6 green apples, so the total is 9.",
  "metadata": {
    "plan": "{\"problem_type\": \"apples\", \"red\": 3, \"green_multiplier\": 2, \"steps\": [\"Extract the number of red apples.\", \"Calculate the number of green apples.\", \"Add the two quantities.\"]}",
    "checks": [
      {
        "check_name": "independent_calculation_and_constraints",
        "passed": false,
        "details": "Expected total: 9 apples."
      },
      {
        "check_name": "independent_calculation_and_constraints",
        "passed": true,
        "details": "Expected total: 9 apples."
      }
    ],
    "retries": 1
  }
}
Retry test passed!


In [30]:

readme = r"""
# Multi-Step Reasoning Agent with Self-Checking

## Overview
A Python notebook-based agent that solves supported arithmetic and time
word problems using three separate phases: Planner, Executor, and Verifier.

## Architecture
1. **Planner** extracts problem facts and creates a structured plan.
2. **Executor** computes a proposed answer and records key calculations.
3. **Verifier** independently recalculates the result and checks constraints.
4. **Retry loop** repeats execution when verification fails, up to two retries.
5. **JSON output** returns the answer, status, concise explanation, plan,
   verification checks, and retry count.

## Requirements
- Python 3
- Google Colab or a local Python environment
- Standard library only for the offline baseline

## How to run
1. Open `Multi_Step_Reasoning_Agent.ipynb` in Google Colab.
2. Run each cell from top to bottom.
3. Call `solve("your question here")`.
4. Run the evaluation suite to produce `example_run_logs.json`.

## Where prompts live
`PLANNER_PROMPT`, `EXECUTOR_PROMPT`, and `VERIFIER_PROMPT` are separate
templates near the start of the notebook. They document the intended LLM
interface and include examples for each phase.

The submitted baseline uses deterministic Python functions rather than
live LLM calls, so it can be tested without API credentials or quota.
The prompt templates are documentation for a future LLM backend and are
not executed by the offline implementation.

## Supported problem types
- Apple totals where green apples are twice the red apples
- Multi-step chocolate totals
- Item quantity multiplied by unit price
- Journey time differences, including journeys crossing midnight
- Selecting meeting slots long enough for a required duration

Unsupported question formats fail safely instead of guessing.

## Output schema
The `solve(question)` function returns a dictionary with:
- `answer`
- `status`
- `reasoning_visible_to_user`
- `metadata.plan`
- `metadata.checks`
- `metadata.retries`

The user-facing explanation is short. Detailed internal chain-of-thought
is not exposed.

## Evaluation
The notebook includes eight easy tests and four tricky tests, plus a
deliberate incorrect-answer test to demonstrate retry behavior.
Test logs are saved to `example_run_logs.json`.

## Prompt design
Each phase has a separate responsibility and a specified JSON structure.
Examples help define expected behavior and reduce output-format ambiguity.
The offline version uses deterministic rules so calculations and checks
are reproducible.

## Limitations and assumptions
- The baseline supports only the explicitly implemented question patterns.
- Time values use 24-hour `HH:MM` format.
- Overnight journeys are supported; meeting slots are assumed to start
  and end on the same day.
- The verifier shares the Planner's extracted facts, so incorrect parsing
  can affect both execution and verification.
- Prompt templates are not live model calls in this version.

## What did not work well
The initial Gemini-based prototype reached the free-tier daily API quota.
This prevented further live-model tests. The offline implementation was
built to continue development and validate the agent loop without API calls.

## Future improvements
- Add an actual LLM backend behind a configurable interface.
- Validate and retry malformed model JSON.
- Add broader arithmetic, logic, and constraint parsing.
- Use independent parsing or a second verifier to reduce shared errors.
- Add unit tests, CI, and more adversarial test cases.
"""

Path("README.md").write_text(readme.strip() + "\n", encoding="utf-8")

print("README.md created successfully.")


README.md created successfully.


In [31]:

from google.colab import files

files.download("README.md")
files.download("example_run_logs.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>